# Weekly Analytics Brief — OT — Data Pull

Builds the numbers for the weekly brief. Sections, in build order:
1. **Acquisitions & Churn** (from the daily sheet) — OLJ vs OT, net subscription change
2. GA4 (users, sessions, page views, top articles) — *not built yet*
3. CMS (top articles detail, if GA4 doesn't cover it) — *not built yet*

Run top to bottom. The last cell prints the numbers in the same shape as the weekly brief template,
so you can eyeball it before we wire it into the doc.


## Dependencies

Run this once per Colab session (installs are wiped when the runtime resets).

In [1]:
!pip install -q google-api-python-client google-auth google-auth-oauthlib google-analytics-data python-docx


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import pandas as pd
import datetime as dt

# --- Week window ---
# Auto-detects the most recently COMPLETED Monday-Sunday week based on today's date.
# To check a specific past week instead, uncomment the override line below and edit it.
today = dt.datetime.combine(dt.date.today(), dt.time.min)
WEEK_END = today - dt.timedelta(days=today.weekday() + 1)  # most recent Sunday before today
# WEEK_END = dt.datetime(2026, 9, 20)  # <- uncomment + edit to override with a specific week

WEEK_START = WEEK_END - dt.timedelta(days=6)  # the Monday of that week
PREV_WEEK_END = WEEK_START - dt.timedelta(days=1)
PREV_WEEK_START = PREV_WEEK_END - dt.timedelta(days=6)

assert WEEK_END.weekday() == 6, "WEEK_END must be a Sunday"

print(f"This week:  {WEEK_START:%Y-%m-%d} (Mon) → {WEEK_END:%Y-%m-%d} (Sun)")
print(f"Last week:  {PREV_WEEK_START:%Y-%m-%d} (Mon) → {PREV_WEEK_END:%Y-%m-%d} (Sun)")


This week:  2026-09-14 (Mon) → 2026-09-20 (Sun)
Last week:  2026-09-07 (Mon) → 2026-09-13 (Sun)


## 0. Read the source sheet (read-only)

Reads values straight from the live Google Sheet via the Sheets API — **no file is downloaded, and
nothing is ever written back**. Uses the `spreadsheets.readonly` OAuth scope, which has no write
methods at all, so there's no code path here that could edit the sheet even by mistake.

**One-time setup** (not part of the weekly run):
1. In Google Cloud Console, create a **service account** and download its JSON key.
2. Share the Google Sheet with that service account's email (looks like
   `something@project-id.iam.gserviceaccount.com`) as **Viewer**.
3. For GitHub Actions: store the JSON key's contents as a repo secret (e.g. `GSHEET_SA_KEY`); the
   workflow writes it to `service_account.json` at run time before this notebook runs. Locally, just
   save the key as `service_account.json` next to this notebook.


In [3]:
import os

SPREADSHEET_ID = "11WU-b3nmvyPlO0fX9VycKgObr-v5-hXTN6ieCv2TOoA"
SERVICE_ACCOUNT_FILE = "service_account.json"

def read_sheet_rows_api(spreadsheet_id, sheet_name, service_account_file, last_col="R", last_row=5000):
    """Reads a tab's rows (from column B, row 3, to last_col/last_row) via the Sheets API.
    Read-only scope — .readonly has no update/append/clear methods, so this can only ever read.
    Returns a list of rows; each row is a list of raw cell values (UNFORMATTED_VALUE, so dates come
    back as serial numbers, same convention Excel uses)."""
    from google.oauth2 import service_account
    from googleapiclient.discovery import build

    creds = service_account.Credentials.from_service_account_file(
        service_account_file,
        scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],  # read-only, no write methods exist on this scope
    )
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{sheet_name}'!B3:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE",
    ).execute()
    return result.get("values", [])

def read_sheet_rows_local(path, sheet_name):
    """Fallback for local testing when service_account.json isn't set up yet: reads the same
    B3:R-range shape out of the uploaded snapshot file, so downstream code is identical either way."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[sheet_name]
    rows = []
    for r in range(3, ws.max_row + 1):
        row = [ws.cell(row=r, column=c).value for c in range(2, 19)]  # B..R
        if any(v is not None for v in row):
            rows.append(row)
    return rows

USE_API = os.path.exists(SERVICE_ACCOUNT_FILE)
if USE_API:
    print("Reading live sheet via Sheets API (read-only)...")
else:
    print("[no service_account.json found — reading local snapshot instead for now]")


[no service_account.json found — reading local snapshot instead for now]


## 1. Acquisitions & Churn

Rows come from section 0 above (live via API, or local snapshot as fallback) — column indices below
are relative to column B (B=0), matching the B:R range fetched there.

Column layout (confirmed against the sheet's own pre-built totals, so we trust these instead of
re-summing raw columns by hand):
- **Acquisitions**: O = grand total, P = OLJ Basic total, Q = OLJ Premium total, R = OT total → OLJ = P+Q
- **Churns**: L = grand total, M = OLJ Basic total, N = OLJ Premium total, O = OT total → OLJ = M+N


In [4]:
def excel_serial_to_datetime(v):
    """API returns dates as Excel-style serial numbers (UNFORMATTED_VALUE); openpyxl local fallback
    already gives real datetimes. Handle both."""
    if isinstance(v, dt.datetime):
        return v
    if isinstance(v, (int, float)):
        return dt.datetime(1899, 12, 30) + dt.timedelta(days=v)
    return None

def week_totals_from_rows(rows, col_idxs, start, end):
    """col_idxs are 0-based, relative to column B (date is index 0)."""
    totals = {c: 0 for c in col_idxs}
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            for c in col_idxs:
                v = row[c] if c < len(row) else 0
                totals[c] += v or 0
    return totals

# 0-based offsets from column B: O=13, P=14, Q=15, R=16 (Acquisitions); L=10, M=11, N=12, O=13 (Churns)
ACQ_COLS = [13, 14, 15, 16]
CHU_COLS = [10, 11, 12, 13]

def acq_churn_week(start, end):
    if USE_API:
        acq_rows = read_sheet_rows_api(SPREADSHEET_ID, "Acquisitions", SERVICE_ACCOUNT_FILE)
        chu_rows = read_sheet_rows_api(SPREADSHEET_ID, "Churns", SERVICE_ACCOUNT_FILE)
    else:
        acq_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Acquisitions")
        chu_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Churns")

    acq = week_totals_from_rows(acq_rows, ACQ_COLS, start, end)
    chu = week_totals_from_rows(chu_rows, CHU_COLS, start, end)

    olj_new = acq[14] + acq[15]
    ot_new  = acq[16]
    olj_churn = chu[11] + chu[12]
    ot_churn  = chu[13]

    return {
        "olj_new": olj_new, "olj_churn": olj_churn, "olj_net": olj_new - olj_churn,
        "ot_new": ot_new,   "ot_churn": ot_churn,   "ot_net": ot_new - ot_churn,
    }

this_week = acq_churn_week(WEEK_START, WEEK_END)
last_week = acq_churn_week(PREV_WEEK_START, PREV_WEEK_END)

this_week, last_week


({'olj_new': 64,
  'olj_churn': 104,
  'olj_net': -40,
  'ot_new': 25,
  'ot_churn': 13,
  'ot_net': 12},
 {'olj_new': 60,
  'olj_churn': 87,
  'olj_net': -27,
  'ot_new': 24,
  'ot_churn': 20,
  'ot_net': 4})

In [5]:
def net_row(this_w, last_w, prefix):
    return {
        "This week": f"{this_w[f'{prefix}_net']:+d} ({this_w[f'{prefix}_new']} new / {this_w[f'{prefix}_churn']} churn)",
        "Last week": f"{last_w[f'{prefix}_net']:+d} ({last_w[f'{prefix}_new']} new / {last_w[f'{prefix}_churn']} churn)",
        "WoW (Δ net)": f"{this_w[f'{prefix}_net'] - last_w[f'{prefix}_net']:+d}",
    }

olj_table = pd.DataFrame([net_row(this_week, last_week, "olj")], index=["Net Subscription Change (new − churn)"])
ot_table  = pd.DataFrame([net_row(this_week, last_week, "ot")],  index=["Net Subscription Change (new − churn)"])

print("OLJ")
display(olj_table)
print("\nOT")
display(ot_table)


OLJ


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),-40 (64 new / 104 churn),-27 (60 new / 87 churn),-13



OT


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),+12 (25 new / 13 churn),+4 (24 new / 20 churn),+8


## 2. New accounts (OLJ vs OT)

Simpler than the CMS scraping approach — this is already tracked daily in the **`Accounts created OLJ OT`**
tab of the same spreadsheet (columns: Date, Accounts created in period OT, Accounts created in period OLJ).
Reuses the exact same read-only connection from section 0 — no separate login needed.

Note: this whole report is **OLJ-specific**, matching the GA4 stream filter — OT is kept alongside for
reference since the sheet already tracks both, but OLJ is the number that matters for the brief.


In [6]:
def read_accounts_created_rows():
    if USE_API:
        return read_sheet_rows_api_custom(SPREADSHEET_ID, "Accounts created OLJ OT",
                                            SERVICE_ACCOUNT_FILE, first_col="A", last_col="D")
    else:
        import openpyxl
        wb = openpyxl.load_workbook("Daily_sheet_Acquisitions_Churns.xlsx", data_only=True)
        ws = wb["Accounts created OLJ OT"]
        rows = []
        for r in range(2, ws.max_row + 1):
            row = [ws.cell(row=r, column=c).value for c in range(1, 5)]  # A..D
            if any(v is not None for v in row):
                rows.append(row)
        return rows

def read_sheet_rows_api_custom(spreadsheet_id, sheet_name, service_account_file, first_col="A", last_col="D", last_row=5000):
    """Same idea as read_sheet_rows_api in section 0, but this tab\'s date column is A, not B,
    so it needs its own starting column."""
    from google.oauth2 import service_account as sa
    from googleapiclient.discovery import build

    creds = sa.Credentials.from_service_account_file(
        service_account_file, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"\'{sheet_name}\'!{first_col}2:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE",
    ).execute()
    return result.get("values", [])

def accounts_created_week(start, end):
    rows = read_accounts_created_rows()
    ot_total, olj_total = 0, 0
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            ot_total += (row[1] if len(row) > 1 and row[1] else 0)
            olj_total += (row[2] if len(row) > 2 and row[2] else 0)
    return {"olj_new_accounts": int(olj_total), "ot_new_accounts": int(ot_total)}

this_week_accounts = accounts_created_week(WEEK_START, WEEK_END)
last_week_accounts = accounts_created_week(PREV_WEEK_START, PREV_WEEK_END)

this_week_accounts, last_week_accounts


({'olj_new_accounts': 0, 'ot_new_accounts': 0},
 {'olj_new_accounts': 68, 'ot_new_accounts': 14})

## 3. GA4 — Users, Sessions, Page views, Top articles, Countries, Sources

Uses `ot_stream_filter` throughout below. `olj_stream_filter` is also defined in this cell (shared setup code) but unused in this dedicated OT script.


In [7]:
GA4_PROPERTY_ID = "328439412"  # GA4 Admin > Property Details
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"  # from Cloud Console > Credentials > OAuth client ID (Desktop app)
OAUTH_TOKEN_FILE = "token.json"  # created automatically after your first approval; reused silently after that
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    """Prefers the service account (once Property Access Management is granted); falls back to
    logging in as you via OAuth, which only needs a browser click on the very first run."""
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account as ga_service_account
        return ga_service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)

    if os.path.exists(OAUTH_CLIENT_SECRET_FILE):
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow

        creds = None
        if os.path.exists(OAUTH_TOKEN_FILE):
            creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES)
                creds = flow.run_local_server(port=0)  # opens your browser -- click Allow once
            with open(OAUTH_TOKEN_FILE, "w") as f:
                f.write(creds.to_json())
        return creds

    return None

GA4_READY = (
    (os.path.exists(SERVICE_ACCOUNT_FILE) or os.path.exists(OAUTH_CLIENT_SECRET_FILE))
    and GA4_PROPERTY_ID != "REPLACE_WITH_YOUR_PROPERTY_ID"
)

# Always defined (even as None when GA4 isn't connected yet), so downstream cells can safely pass
# stream_filter=olj_stream_filter / ot_stream_filter regardless of GA4_READY -- avoids a NameError
# in the example-data fallback path, where these never get built.
olj_stream_filter = None
ot_stream_filter = None

if GA4_READY:
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        RunReportRequest, DateRange, Metric, Dimension, OrderBy,
        FilterExpression, Filter,
    )
    ga4_creds = get_ga4_credentials()
    ga4_client = BetaAnalyticsDataClient(credentials=ga4_creds)

    # Every OLJ table in the report is filtered to OLJ streams (name contains "olj",
        # case-insensitive, so it matches "OLJ Website" / "olj android" / "OLJ iOS" etc.)
    olj_stream_filter = FilterExpression(
        filter=Filter(
            field_name="streamName",
            string_filter=Filter.StringFilter(
                match_type=Filter.StringFilter.MatchType.CONTAINS,
                value="olj",
                case_sensitive=False,
            )
        )
    )

    # OT streams don't share one consistent naming pattern -- "OT iOS"/"OT Android" start with
    # "OT", but the web stream is apparently named "L'Orient Today" with no "OT" in it at all.
    # A bare CONTAINS "ot" would be too loose (matches almost anything with those two letters
    # together), so this ORs two specific, safe patterns instead: starts with "OT", OR contains
    # "orient today".
    from google.analytics.data_v1beta.types import FilterExpressionList
    ot_stream_filter = FilterExpression(
        or_group=FilterExpressionList(expressions=[
            FilterExpression(filter=Filter(
                field_name="streamName",
                string_filter=Filter.StringFilter(
                    match_type=Filter.StringFilter.MatchType.BEGINS_WITH,
                    value="OT",
                    case_sensitive=False,
                )
            )),
            FilterExpression(filter=Filter(
                field_name="streamName",
                string_filter=Filter.StringFilter(
                    match_type=Filter.StringFilter.MatchType.CONTAINS,
                    value="orient today",
                    case_sensitive=False,
                )
            )),
        ])
    )

    def ga4_report(start, end, metrics, dimensions=None, order_by_metric=None, limit=None,
                   extra_filter=None, stream_filter=None):
        base_filter = stream_filter if stream_filter is not None else olj_stream_filter
        dim_filter = base_filter
        if extra_filter is not None:
            from google.analytics.data_v1beta.types import FilterExpressionList
            dim_filter = FilterExpression(
                and_group=FilterExpressionList(expressions=[base_filter, extra_filter])
            )
        request = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            date_ranges=[DateRange(start_date=start.strftime("%Y-%m-%d"), end_date=end.strftime("%Y-%m-%d"))],
            metrics=[Metric(name=m) for m in metrics],
            dimensions=[Dimension(name=d) for d in (dimensions or [])],
            dimension_filter=dim_filter,
            limit=limit,
        )
        if order_by_metric:
            request.order_bys = [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=order_by_metric), desc=True)]
        resp = ga4_client.run_report(request)
        cols = [d.name for d in resp.dimension_headers] + [m.name for m in resp.metric_headers]
        rows = []
        for row in resp.rows:
            rows.append([v.value for v in row.dimension_values] + [v.value for v in row.metric_values])
        return pd.DataFrame(rows, columns=cols)

    print(f"GA4 ready -- property {GA4_PROPERTY_ID}")
else:
    print("[GA4 not connected yet -- using example data for this section. "
          "Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]")


[GA4 not connected yet -- using example data for this section. Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]


In [8]:
def ga4_scorecard(start, end, stream_filter=None):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["totalUsers", "sessions", "screenPageViews"], stream_filter=stream_filter)
        row = df.iloc[0]
        return {"users": int(row["totalUsers"]), "sessions": int(row["sessions"]), "page_views": int(row["screenPageViews"])}
    else:
        # Example data, roughly matching the shape from the template we built earlier
        import random
        random.seed(hash(start))
        base = 42000 if start == WEEK_START else 39900
        return {"users": base, "sessions": int(base * 1.39), "page_views": int(base * 2.66)}

def wow_pct(this_v, last_v):
    if last_v == 0:
        return "n/a"
    pct = (this_v - last_v) / last_v * 100
    arrow = "\u25b2" if pct >= 0 else "\u25bc"
    return f"{arrow} {abs(pct):.0f}%"

def signed_wow_pct(this_v, last_v):
    """For metrics that can be negative (like Net Subscription Change): direction is
    based on whether the value actually improved (this_v >= last_v), not the raw sign of
    the % change -- plain division misleads here, since -27 -> -40 is worse but naive
    math (dividing two negatives) would show it as a positive-looking \"+48%\"."""
    if last_v == 0:
        return "n/a"
    pct = abs(this_v - last_v) / abs(last_v) * 100
    arrow = "\u25b2" if this_v >= last_v else "\u25bc"
    return f"{arrow} {pct:.0f}%"


In [ ]:
import re

ARTICLE_ID_PATTERN = re.compile(r"^\d{6,7}$")  # valid article IDs only -- drops "(not set)" and junk

def _top2_sources(rows):
    """rows: list of (source, views) for one article. Returns two 'source NN%' strings, where the
    percentage is that source's share of THIS article's views (not overall site traffic)."""
    total = sum(v for _, v in rows)
    if total == 0:
        return "", ""
    ranked = sorted(rows, key=lambda t: t[1], reverse=True)[:2]
    out = [f"{src} {round(v / total * 100)}%" for src, v in ranked]
    while len(out) < 2:
        out.append("")
    return out[0], out[1]

def ga4_top_articles_by_id(start, end, n=10, stream_filter=None):
    if GA4_READY:
        # pageTitle + sessionSource pulled in the SAME query as the ID -- not a separate lookup call
        web = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:articleid", "sessionSource", "pageTitle"], stream_filter=stream_filter)
        app = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:article_id", "sessionSource", "pageTitle"], stream_filter=stream_filter)

        web = web.rename(columns={"customEvent:articleid": "article_id", "screenPageViews": "views"})
        app = app.rename(columns={"customEvent:article_id": "article_id", "screenPageViews": "views"})

        combined = pd.concat([web, app], ignore_index=True)
        combined["views"] = combined["views"].astype(int)

        # Keep only valid numeric article IDs (6 or 7 digits) -- excludes "(not set)", blanks, junk
        valid = combined["article_id"].astype(str).str.strip().apply(lambda v: bool(ARTICLE_ID_PATTERN.match(v)))
        combined = combined[valid]

        totals = combined.groupby("article_id", as_index=False)["views"].sum()
        top = totals.sort_values("views", ascending=False).head(n).reset_index(drop=True)

        # Rank by ID first (above), THEN attach a title per ID -- the most frequent title seen for
        # that ID across both web and app rows (guards against a stray differently-formatted title)
        subset = combined[combined["article_id"].isin(top["article_id"])]
        titles = (
            subset.groupby("article_id")["pageTitle"]
            .agg(lambda s: s.value_counts().idxmax() if len(s) else "")
            .to_dict()
        )

        # Top 2 traffic sources per article, as a share of that article's own views
        source_totals = subset.groupby(["article_id", "sessionSource"])["views"].sum()
        src1, src2 = {}, {}
        for aid in top["article_id"]:
            rows = list(source_totals.loc[aid].items()) if aid in source_totals.index.get_level_values(0) else []
            s1, s2 = _top2_sources(rows)
            src1[aid], src2[aid] = s1, s2

        top["Article"] = top["article_id"].map(titles)
        top["Top source 1"] = top["article_id"].map(src1)
        top["Top source 2"] = top["article_id"].map(src2)
        return top.rename(columns={"article_id": "Article ID", "views": "Views"})[
            ["Article", "Article ID", "Views", "Top source 1", "Top source 2"]
        ]
    else:
        example_sources = [
            ("google 54%", "(direct) 22%"), ("facebook 41%", "google 30%"),
            ("(direct) 38%", "newsletter 19%"), ("google 47%", "instagram 15%"),
            ("(direct) 33%", "google 28%"), ("google 39%", "bing 12%"),
            ("newsletter 44%", "(direct) 21%"), ("google 36%", "facebook 20%"),
            ("(direct) 29%", "google 24%"), ("google 31%", "(direct) 26%"),
        ][:n]
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Article ID": [f"15481{i:02d}" for i in range(1, n + 1)],
            "Views": sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n],
            "Top source 1": [s[0] for s in example_sources],
            "Top source 2": [s[1] for s in example_sources],
        })


In [ ]:
def ga4_top_dimension(start, end, dimension, n=5, stream_filter=None):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["sessions"], dimensions=[dimension],
                         order_by_metric="sessions", limit=n, stream_filter=stream_filter)
        df["sessions"] = df["sessions"].astype(int)
        total = df["sessions"].sum()
        df["share"] = (df["sessions"] / total * 100).round(0).astype(int).astype(str) + "%"
        return df.rename(columns={dimension: dimension.capitalize(), "sessions": "Sessions", "share": "Share"})
    else:
        # Example data varies slightly by week so the WoW comparison below has something to show
        if dimension == "country":
            if start == WEEK_START:
                data = {"Country": ["Lebanon", "France", "USA", "Canada", "UAE"],
                        "Sessions": [24800, 8900, 6200, 3100, 2400]}
            else:
                data = {"Country": ["Lebanon", "France", "USA", "UAE", "Canada"],
                        "Sessions": [23100, 9400, 5800, 2600, 2200]}
            df = pd.DataFrame(data).head(n)
            df["Share"] = (df["Sessions"] / df["Sessions"].sum() * 100).round(0).astype(int).astype(str) + "%"
            return df
        return pd.DataFrame()

def top_countries_with_wow(n=5, stream_filter=None):
    """Top N countries this week, with last week's share AND a WoW % alongside for the same
    countries. Last week is pulled from a wider slice (not just its own top N) so a country that's
    top-N this week but wasn't top-N last week can still be matched and compared; WoW is computed
    from the underlying session counts, not the rounded share string, so it stays accurate."""
    this_c = ga4_top_dimension(WEEK_START, WEEK_END, "country", n=n, stream_filter=stream_filter)
    last_c_wide = ga4_top_dimension(PREV_WEEK_START, PREV_WEEK_END, "country", n=max(n * 4, 20), stream_filter=stream_filter)

    last_share = dict(zip(last_c_wide["Country"], last_c_wide["Share"]))
    last_sessions = dict(zip(last_c_wide["Country"], last_c_wide["Sessions"]))

    this_c = this_c.rename(columns={"Share": "This week"})
    this_c["Last week"] = this_c["Country"].map(last_share).fillna("—")
    this_c["WoW"] = this_c.apply(
        lambda r: wow_pct(r["Sessions"], last_sessions[r["Country"]]) if r["Country"] in last_sessions else "n/a",
        axis=1,
    )
    return this_c[["Country", "Last week", "This week", "WoW"]]


In [ ]:
MAIN_CATS = ["Direct", "Search Engines", "Social Networks", "AI Assistants", "Internal/Newsletters", "Other"]

MAIN_COLORS = {
    "Direct":               "#4285F4",
    "Search Engines":       "#00BFA5",
    "Social Networks":      "#9C27B0",
    "AI Assistants":        "#FFA726",
    "Internal/Newsletters": "#FF6B9D",
    "Other":                "#9E9E9E",
}

DIRECT = {"(direct)"}

SEARCH_ENGINES = {
    "google", "news.google.com", "bing", "ecosia.org", "qwant.com", "duckduckgo",
    "fr.search.yahoo.com", "yahoo", "search.brave.com", "yandex", "ya.ru", "startpage.com",
}

AI_ASSISTANTS = {
    "chatgpt.com", "perplexity.ai", "gemini.google.com", "perplexity", "copilot.com",
    "copilot.microsoft.com", "openai", "duck.ai", "chat.mistral.ai",
    "notebooklm.google.com", "claude.ai", "poe.com", "grok.com", "chat.qwen.ai",
    "doubao.com", "chat.z.ai", "felo.ai", "mammouth.ai", "you.com",
}

SOCIAL_NETWORKS_OT = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-48", "reddit.com", "old.reddit.com", "out.reddit.com",
    "fb", "flipboard", "flipboard.com",
    # Note: WhatsApp is "cms-48" for OT (it was "cms-46" for OLJ).
}

INTERNAL_NEWSLETTERS_OT = {
    "mailchimp", "newsletter", "email", "website", "ot", "olj", "google-play",
    "ot.me", "olj.me", "autopromoot", "autopromoolj", "morningbrief", "hs_email",
    "brevo", "mailchi.mp", "us1.campaign-archive.com", "actito.be", "activetrail",
    "acumbamail", "omnisend", "wordfly", "newsletter_1", "newsletter_6b",
    "newsletter_paiementechouepp", "newsletter_preventif", "partenairesjamhour",
    "marketo", "gmi mailchimp integration prod list", "master list", "bundle", "nb",
    # CMS-34 = Morning Brief, CMS-9 = A la une (both OT-owned newsletter/homepage widgets):
    "cms-34", "cms-9",
}

def categorize_source(src, social=SOCIAL_NETWORKS_OT, newsletters=INTERNAL_NEWSLETTERS_OT):
    s = str(src).strip().lower()
    if s in DIRECT:
        return "Direct"
    if s in SEARCH_ENGINES:
        return "Search Engines"
    if s in social:
        return "Social Networks"
    if s in AI_ASSISTANTS:
        return "AI Assistants"
    if s in newsletters:
        return "Internal/Newsletters"
    return "Other"

def ga4_sources_by_category(start, end, stream_filter=None, social=SOCIAL_NETWORKS_OT, newsletters=INTERNAL_NEWSLETTERS_OT):
    if GA4_READY:
        # No limit -- need every distinct source to categorize correctly, not just a top-N slice
        df = ga4_report(start, end, metrics=["sessions"], dimensions=["sessionSource"], limit=100000,
                         stream_filter=stream_filter)
        df["sessions"] = df["sessions"].astype(int)
        df["Category"] = df["sessionSource"].apply(lambda s: categorize_source(s, social, newsletters))
        totals = df.groupby("Category")["sessions"].sum().reindex(MAIN_CATS, fill_value=0)
    else:
        # Varies slightly by week so the WoW comparison below has something real to show
        if start == WEEK_START:
            totals = pd.Series(
                {"Direct": 15200, "Search Engines": 13700, "Social Networks": 9400,
                 "AI Assistants": 1200, "Internal/Newsletters": 9400, "Other": 3100}
            ).reindex(MAIN_CATS, fill_value=0)
        else:
            totals = pd.Series(
                {"Direct": 14100, "Search Engines": 12300, "Social Networks": 10200,
                 "AI Assistants": 900, "Internal/Newsletters": 8700, "Other": 2900}
            ).reindex(MAIN_CATS, fill_value=0)

    total = totals.sum()
    result = pd.DataFrame({"Category": MAIN_CATS, "Sessions": totals.values.astype(int)})
    result["Share"] = (result["Sessions"] / total * 100).round(0).astype(int).astype(str) + "%"
    result["Color"] = result["Category"].map(MAIN_COLORS)
    return result

def sources_by_category_with_wow(stream_filter=None, social=SOCIAL_NETWORKS_OT, newsletters=INTERNAL_NEWSLETTERS_OT):
    """This week's category breakdown, with last week's share and a WoW % alongside (WoW computed
    from the underlying session counts, not the rounded share, so it stays accurate)."""
    this_df = ga4_sources_by_category(WEEK_START, WEEK_END, stream_filter=stream_filter, social=social, newsletters=newsletters)
    last_df = ga4_sources_by_category(PREV_WEEK_START, PREV_WEEK_END, stream_filter=stream_filter, social=social, newsletters=newsletters)
    merged = this_df.merge(
        last_df[["Category", "Sessions", "Share"]], on="Category", suffixes=("", " (last)")
    )
    merged["WoW"] = merged.apply(lambda r: wow_pct(r["Sessions"], r["Sessions (last)"]), axis=1)
    merged = merged.rename(columns={"Share": "This week", "Share (last)": "Last week"})
    return merged[["Category", "Last week", "This week", "WoW", "Color"]]


In [12]:
def ga4_app_downloads(start, end, stream_filter=None):
    if GA4_READY:
        from google.analytics.data_v1beta.types import Filter as _Filter, FilterExpression as _FE
        event_filter = _FE(filter=_Filter(field_name="eventName",
                                           string_filter=_Filter.StringFilter(value="app_download")))
        df = ga4_report(start, end, metrics=["eventCount"], dimensions=["platform"],
                         extra_filter=event_filter, stream_filter=stream_filter)
        counts = {row["platform"]: int(row["eventCount"]) for _, row in df.iterrows()}
        return {"ios": counts.get("iOS", 0), "android": counts.get("Android", 0)}
    else:
        return {"ios": 520, "android": 370} if start == WEEK_START else {"ios": 480, "android": 337}


In [13]:
from IPython.display import Markdown, display
from itertools import zip_longest

def pct_change(this_v, last_v):
    if last_v == 0:
        return None
    return (this_v - last_v) / last_v * 100


## 4. Gather the data (OT-filtered)

Acquisitions/Churns (section 1) and New accounts (section 2) already computed `ot_*` fields --
nothing more to fetch there, just used directly below. This step is the GA4 side: same functions
as any OLJ build would use, just called with `ot_stream_filter` and the OT category sets.


In [ ]:
this_week_ga4 = ga4_scorecard(WEEK_START, WEEK_END, stream_filter=ot_stream_filter)
last_week_ga4 = ga4_scorecard(PREV_WEEK_START, PREV_WEEK_END, stream_filter=ot_stream_filter)

top_articles_by_id = ga4_top_articles_by_id(WEEK_START, WEEK_END, stream_filter=ot_stream_filter)

top_countries = top_countries_with_wow(n=5, stream_filter=ot_stream_filter)
sources_by_category = sources_by_category_with_wow(
    stream_filter=ot_stream_filter, social=SOCIAL_NETWORKS_OT, newsletters=INTERNAL_NEWSLETTERS_OT,
)

this_week_downloads = ga4_app_downloads(WEEK_START, WEEK_END, stream_filter=ot_stream_filter)
last_week_downloads = ga4_app_downloads(PREV_WEEK_START, PREV_WEEK_END, stream_filter=ot_stream_filter)

this_week_ga4, last_week_ga4


## 5. Final report — OT Weekly Brief

In [ ]:
dl_this_total = sum(this_week_downloads.values())
dl_last_total = sum(last_week_downloads.values())

movers = {
    "Users": pct_change(this_week_ga4["users"], last_week_ga4["users"]),
    "Sessions": pct_change(this_week_ga4["sessions"], last_week_ga4["sessions"]),
    "Page views": pct_change(this_week_ga4["page_views"], last_week_ga4["page_views"]),
    "App downloads": pct_change(dl_this_total, dl_last_total),
}

biggest_mover = max(movers, key=lambda k: abs(movers[k]) if movers[k] is not None else 0)
mv = movers[biggest_mover]
direction = "up" if mv is not None and mv >= 0 else "down"
headline = f"{biggest_mover} {direction} {abs(mv):.0f}% week-over-week." if mv is not None else "No clear standout metric this week."

watch_lines = []
for name, pct in movers.items():
    if pct is not None and pct <= -10:
        watch_lines.append(f"{name} down {abs(pct):.0f}% week-over-week.")
new_subs_delta = this_week["ot_new"] - last_week["ot_new"]
if new_subs_delta < 0:
    watch_lines.append(f"New OT subscriptions down {abs(new_subs_delta)} vs. last week ({this_week['ot_new']} this week).")
watch_line = " ".join(watch_lines) if watch_lines else "Nothing off-trend this week."

# Column order: Last week before This week. Net Subscription Change (which mixed in churn) is
# dropped in favor of a straight acquisitions count -- new people only, no churn.
scorecard_md = f"""| Metric | Last week | This week | WoW |
|---|---|---|---|
| Users | {last_week_ga4['users']:,} | {this_week_ga4['users']:,} | {wow_pct(this_week_ga4['users'], last_week_ga4['users'])} |
| Sessions | {last_week_ga4['sessions']:,} | {this_week_ga4['sessions']:,} | {wow_pct(this_week_ga4['sessions'], last_week_ga4['sessions'])} |
| Page views | {last_week_ga4['page_views']:,} | {this_week_ga4['page_views']:,} | {wow_pct(this_week_ga4['page_views'], last_week_ga4['page_views'])} |
| New accounts | {last_week_accounts['ot_new_accounts']:,} | {this_week_accounts['ot_new_accounts']:,} | {wow_pct(this_week_accounts['ot_new_accounts'], last_week_accounts['ot_new_accounts'])} |
| New OT subscriptions (acquisitions) | {last_week['ot_new']:,} | {this_week['ot_new']:,} | {wow_pct(this_week['ot_new'], last_week['ot_new'])} |
| App downloads (iOS / Android) | {dl_last_total} ({last_week_downloads['ios']} / {last_week_downloads['android']}) | {dl_this_total} ({this_week_downloads['ios']} / {this_week_downloads['android']}) | {wow_pct(dl_this_total, dl_last_total)} |"""

def _sources_cell(row):
    s = row["Top source 1"]
    if row["Top source 2"]:
        s += f" · {row['Top source 2']}"
    return s

# Full titles here -- this table is the one place we've added color/styling, so it can afford
# the extra width.
articles_md = "| # | Article | Views | Top sources |\n|---|---|---|---|\n" + "\n".join(
    f"| {i+1} | {row['Article']} | {row['Views']:,} | {_sources_cell(row)} |"
    for i, row in top_articles_by_id.reset_index(drop=True).iterrows()
)

countries_md = "| Country | Last week | This week | WoW |\n|---|---|---|---|\n" + "\n".join(
    f"| {row['Country']} | {row['Last week']} | {row['This week']} | {row['WoW']} |"
    for _, row in top_countries.iterrows()
)

sources_md = "| Source category | Last week | This week | WoW |\n|---|---|---|---|\n" + "\n".join(
    f"| {row['Category']} | {row['Last week']} | {row['This week']} | {row['WoW']} |"
    for _, row in sources_by_category.iterrows()
)

report_md = f"""# Weekly Analytics Brief — OT
### Week of {WEEK_START:%b %d}–{WEEK_END:%b %d, %Y}

**Headline:** {headline}

{scorecard_md}

## Top 10 articles this week
{articles_md}

## Top countries (vs. last week)
{countries_md}

## Sources by category (vs. last week)
{sources_md}

## One thing to watch
{watch_line}
"""

display(Markdown(report_md))


## 6. Export to Word + email it

Same Gmail-App-Password pattern as the OLJ script -- separate credentials/setup, run independently.


In [ ]:
import os
import getpass
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

from docx import Document
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import RGBColor

RECIPIENT_EMAIL = "dianafarhat@lorientlejour.com"

ARTICLES_HEADER_COLOR = "2E5F6B"  # deep teal -- the one spot of custom color, on the articles table header

def _shade_cell(cell, hex_color):
    """Fill a table cell's background with a hex color (no '#')."""
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), hex_color)
    tcPr.append(shd)

def _style_header_cell(cell, fill_hex, font_color=RGBColor(0xFF, 0xFF, 0xFF)):
    _shade_cell(cell, fill_hex)
    for p in cell.paragraphs:
        for run in p.runs:
            run.font.bold = True
            run.font.color.rgb = font_color

def build_brief_docx(path):
    doc = Document()
    doc.add_heading("Weekly Analytics Brief — OT", level=0)
    doc.add_paragraph(f"Week of {WEEK_START:%b %d}–{WEEK_END:%b %d, %Y}")

    p = doc.add_paragraph()
    p.add_run("Headline: ").bold = True
    p.add_run(headline)

    # --- Scorecard: Last week before This week; churn dropped for a plain acquisitions count ---
    doc.add_heading("Scorecard", level=2)
    table = doc.add_table(rows=1, cols=4)
    table.style = "Table Grid"
    hdr = table.rows[0].cells
    hdr[0].text, hdr[1].text, hdr[2].text, hdr[3].text = "Metric", "Last week", "This week", "WoW"

    def add_row(metric, last_v, this_v, wow):
        row = table.add_row().cells
        row[0].text, row[1].text, row[2].text, row[3].text = metric, last_v, this_v, wow

    add_row("Users", f"{last_week_ga4['users']:,}", f"{this_week_ga4['users']:,}",
            wow_pct(this_week_ga4['users'], last_week_ga4['users']))
    add_row("Sessions", f"{last_week_ga4['sessions']:,}", f"{this_week_ga4['sessions']:,}",
            wow_pct(this_week_ga4['sessions'], last_week_ga4['sessions']))
    add_row("Page views", f"{last_week_ga4['page_views']:,}", f"{this_week_ga4['page_views']:,}",
            wow_pct(this_week_ga4['page_views'], last_week_ga4['page_views']))
    add_row("New accounts", f"{last_week_accounts['ot_new_accounts']:,}", f"{this_week_accounts['ot_new_accounts']:,}",
            wow_pct(this_week_accounts['ot_new_accounts'], last_week_accounts['ot_new_accounts']))
    add_row("New OT subscriptions (acquisitions)", f"{last_week['ot_new']:,}", f"{this_week['ot_new']:,}",
            wow_pct(this_week['ot_new'], last_week['ot_new']))
    add_row("App downloads (iOS / Android)",
            f"{dl_last_total} ({last_week_downloads['ios']} / {last_week_downloads['android']})",
            f"{dl_this_total} ({this_week_downloads['ios']} / {this_week_downloads['android']})",
            wow_pct(dl_this_total, dl_last_total))

    # --- Top articles: the one table with color -- full titles (not truncated) and a solid
    # colored header row as the "nice detail" ---
    doc.add_heading("Top 10 articles this week", level=2)
    art_table = doc.add_table(rows=1, cols=4)
    art_table.style = "Table Grid"
    ahdr = art_table.rows[0].cells
    ahdr[0].text, ahdr[1].text, ahdr[2].text, ahdr[3].text = "#", "Article", "Views", "Top sources"
    for cell in ahdr:
        _style_header_cell(cell, ARTICLES_HEADER_COLOR)
    for i, row in top_articles_by_id.reset_index(drop=True).iterrows():
        sources = row['Top source 1']
        if row['Top source 2']:
            sources += f" · {row['Top source 2']}"
        r = art_table.add_row().cells
        r[0].text, r[1].text, r[2].text, r[3].text = str(i + 1), row['Article'], f"{row['Views']:,}", sources

    # --- Top countries, with a last-week comparison and WoW ---
    doc.add_heading("Top countries (vs. last week)", level=2)
    countries_table = doc.add_table(rows=1, cols=4)
    countries_table.style = "Table Grid"
    chdr = countries_table.rows[0].cells
    chdr[0].text, chdr[1].text, chdr[2].text, chdr[3].text = "Country", "Last week", "This week", "WoW"
    for _, row in top_countries.iterrows():
        r = countries_table.add_row().cells
        r[0].text, r[1].text, r[2].text, r[3].text = row['Country'], row['Last week'], row['This week'], row['WoW']

    # --- Sources by category, vs. last week -- plain, no color (color is reserved for articles) ---
    doc.add_heading("Sources by category (vs. last week)", level=2)
    sources_table = doc.add_table(rows=1, cols=4)
    sources_table.style = "Table Grid"
    shdr = sources_table.rows[0].cells
    shdr[0].text, shdr[1].text, shdr[2].text, shdr[3].text = "Source category", "Last week", "This week", "WoW"
    for _, row in sources_by_category.iterrows():
        r = sources_table.add_row().cells
        r[0].text, r[1].text, r[2].text, r[3].text = row['Category'], row['Last week'], row['This week'], row['WoW']

    doc.add_heading("One thing to watch", level=2)
    doc.add_paragraph(watch_line)

    doc.save(path)
    return path

def send_email_with_attachment(recipient, subject, body, attachment_path):
    sender = os.environ.get("GMAIL_ADDRESS") or input("Sending Gmail address: ")
    app_password = os.environ.get("GMAIL_APP_PASSWORD") or getpass.getpass("Gmail App Password (not your normal password): ")

    msg = MIMEMultipart()
    msg["From"] = sender
    msg["To"] = recipient
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    with open(attachment_path, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", f'attachment; filename="{os.path.basename(attachment_path)}"')
    msg.attach(part)

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender, app_password)
        server.send_message(msg)

docx_filename = f"Weekly_Analytics_Brief_OT_{WEEK_START:%Y%m%d}_{WEEK_END:%Y%m%d}.docx"

try:
    build_brief_docx(docx_filename)
    print(f"Saved {docx_filename}")

    send_email_with_attachment(
        RECIPIENT_EMAIL,
        f"Weekly Analytics Brief — OT — Week of {WEEK_START:%b %d}",
        f"Attached: this week's OT analytics brief.\n\n{headline}",
        docx_filename,
    )
    print(f"Emailed to {RECIPIENT_EMAIL}")
except Exception as e:
    print(f"Export/email step failed -- paste this error back and I'll adjust.\n{type(e).__name__}: {e}")
